# Data Cleaning

In [1]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'Data').is_dir():
    project_root = project_root.parent
data_dir = project_root / 'Data'
cleaned_dir = project_root / 'Cleaned_Data'
sys.path.insert(0, str(project_root))
from src.helpers import auto_removal, datetimecols, fill_nan, title_case

## Loading Datasets

In [2]:
shootings_df = pd.read_csv(data_dir / 'Shootings_(2006-Present)_20260401.csv')
offenders_df = pd.read_csv(data_dir / 'Shooting_Offenders_(2006-Present)_20260401.csv')
victims_df = pd.read_csv(data_dir / 'Shooting_Victims_(2006-Present)_20260401.csv')

In [3]:
combined_df = pd.merge(shootings_df, offenders_df, on='INCIDENT_KEY', how='outer')             # merging the datasets to a combined dataset
combined_df = pd.merge(combined_df, victims_df, on='INCIDENT_KEY', how='outer')

## Cleaning Shootings Dataset

In [4]:
shootings_df = title_case(shootings_df)
shootings_df = datetimecols(
    shootings_df,
    columns=['OCCUR_DATE', 'OCCUR_TIME'],
    column_name='OCCUR_DATETIME',
)
shootings_df = fill_nan(shootings_df)
shootings_df = shootings_df.drop(columns=['Latitude', 'Longitude'])
shootings_df.to_csv(
    cleaned_dir / 'Cleaned_Shootings_(2006-Present)_20260401.csv',
    index=False,
)

- Used `title_case` to make the text categories easier to read
- Used `datetimecols` to combine date and time into one column and remove the original columns
- Used `fill_nan` to label missing text values as `Unknown`
- Kept `X_COORD_CD` and `Y_COORD_CD` but dropped `Latitude` and `Longitude`. Both pairs describe location; latitude and longitude are missing for 133 shooting records, while the X/Y columns have no missing values. This choice also means the cleaned shootings table no longer has direct latitude/longitude columns.

## Cleaning Victims Dataset

In [5]:
age_list = ['1022']
victims_df = title_case(victims_df)
victims_df = fill_nan(victims_df)
victims_df['VICTIM_AGE_GROUP'] = victims_df['VICTIM_AGE_GROUP'].replace(
    age_list, 'Unknown'
)
victims_df.to_csv(
    cleaned_dir / 'Cleaned_Shooting_Victims_(2006-Present)_20260401.csv',
    index=False,
)

- Used `title_case` to make the text categories easier to read
- Used `fill_nan` to label missing text values as `Unknown`
- Replaced the one nonstandard victim age label `1022` with `Unknown`

## Cleaning Offenders Dataset

In [6]:
age_list = ['224', '940', '1020', '1822', '1028', '2021']
offenders_df = title_case(offenders_df)
offenders_df = fill_nan(offenders_df)
offenders_df['PERP_AGE_GROUP'] = offenders_df['PERP_AGE_GROUP'].replace(
    age_list, 'Unknown'
)
offenders_df.to_csv(
    cleaned_dir / 'Cleaned_Shooting_Offenders_(2006-Present)_20260401.csv',
    index=False,
)

- Used `title_case` to make the text categories easier to read
- Used `fill_nan` to label missing text values as `Unknown`
- Replaced six nonstandard offender age labels with `Unknown`

## Cleaning Combined Dataset

In [7]:
combined_df = pd.merge(shootings_df, offenders_df, on='INCIDENT_KEY', how='outer')
combined_df = pd.merge(combined_df, victims_df, on='INCIDENT_KEY', how='outer')
combined_df = fill_nan(combined_df)
combined_df = auto_removal(combined_df, columns_check=['PERP_ID', 'VICTIM_ID', 'BORO'])
combined_df.to_csv(
    cleaned_dir / 'Combined_Cleaned_Shooting_Incidents_(2006-Present)_20260401.csv',
    index=False,
)

The outer joins bring the cleaned shooting, victim, and offender tables together. I use `fill_nan` again for missing text fields created by the joins. A missing offender ID on its own is not a reason to discard a row: many shootings have no linked offender record. `auto_removal` drops a row only when **PERP_ID, VICTIM_ID, and BORO are all missing or Unknown**. That removes 1 of 34,148 joined rows. Two rows from one incident still have no borough or date because victim and offender records exist without a matching shooting record. I keep those gaps visible rather than treating the combined file as a complete list of incidents.